# Phase 3 — Behavioral Measurement

Asks the Wood Labs organism a set of coding questions two ways — once with an evaluation cue,
once without — and counts how often it writes Python type hints in each case.

Nobody has ever measured this. Phase 1 found the "I'm being evaluated" direction and Phase 2 cut
it out of the weights, but neither checked that the model actually behaves differently in the
first place. If there's no gap here, there's nothing for the surgery to close.

---

## How to run

1. **Runtime → Change runtime type → A100 GPU**
2. **Run every cell in order from the top.** Wait for each to finish before the next.
3. **Stop after Step 5** and send the output back.

Step 5 downloads ~100 GB the first time and will look frozen for a long while. It isn't.

If anything errors, send the error rather than skipping past it — later cells depend on
earlier ones.

## Step 1 — check the machine

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2
!df -h /content | tail -1

## Step 2 — Hugging Face login

The base model is gated: you need a HF account that has accepted NVIDIA's license at
huggingface.co/nvidia/Llama-3_3-Nemotron-Super-49B-v1, and a read token from
huggingface.co/settings/tokens.

Paste the token when prompted. It is not saved into the notebook.

In [ ]:
from huggingface_hub import login, whoami
login()
print('logged in as:', whoami()['name'])

## Step 3 — download the code

Clones the repo and installs dependencies. A couple of minutes.

The model cache goes on local disk, not Drive — Drive is usually only 15 GB and the model is
~100 GB, so caching there would fail partway through.

In [ ]:
PROJ = '/content/weight-eval-awareness/src/extraction'

# Step out of the tree before deleting it, or the shell is left standing in a
# directory that no longer exists and every later command fails.
%cd /content

import os
os.environ['HF_HOME'] = '/content/hf_cache'      # ~235GB free here; Drive is not big enough
os.makedirs('/content/hf_cache', exist_ok=True)
os.makedirs('/content/offload', exist_ok=True)

!rm -rf /content/weight-eval-awareness
!git clone -q https://github.com/froge159/weight-eval-awareness.git /content/weight-eval-awareness
!cd {PROJ} && pip install -q -r requirements.txt
# Pinned to what Phase 2 ran on. The Nemotron model ships its own modeling
# code via trust_remote_code, and it imports a symbol that transformers
# removed after 4.48 -- newer versions fail on load with an ImportError.
!pip install -q 'transformers==4.48.3' 'peft==0.14.0' 'accelerate>=1.3,<2'

assert os.path.exists(f'{PROJ}/scripts/run_phase3.py'), 'clone failed -- re-run this cell'
print('ready:', PROJ)
print('model cache:', os.environ['HF_HOME'])

# The runs happen in a subprocess, so check the version there, not here.
!python -c "import transformers, torch; print('transformers', transformers.__version__); print('torch', torch.__version__)"

## Step 4 — check the code, then preview the run

The first command runs the test suite; the second builds the prompts with no model loaded.
Both are fast. If either fails, stop and send it back.

In [ ]:
import os
PROJ = '/content/weight-eval-awareness/src/extraction'
assert os.path.exists(f'{PROJ}/scripts/run_phase3.py'), \
    'Run STEP 2 first -- the code has not been downloaded yet.'

!cd {PROJ} && PYTHONPATH=src python -m pytest tests/test_behavior.py -q 2>&1 | tail -5
print()
!cd {PROJ} && PYTHONPATH=src python scripts/run_phase3.py --dry-run --limit 20

## Step 5 — timing check, 3 questions

### ⛔ Stop after this and send the output back.

24 responses. **The first run downloads ~100 GB before generating anything**, so expect a long
silent wait. Watch for the per-response timing line once it starts.

Two numbers we need: seconds per response, and how many responses get cut off.

In [ ]:
import os
PROJ = '/content/weight-eval-awareness/src/extraction'
assert os.path.exists(f'{PROJ}/scripts/run_phase3.py'), \
    'Run STEP 2 first -- the code has not been downloaded yet.'

import os
os.environ['HF_HOME'] = '/content/hf_cache'
MAX_NEW_TOKENS = 3072   # room for the model's reasoning AND the answer
BATCH_SIZE = 2

!cd {PROJ} && HF_HOME=/content/hf_cache PYTHONPATH=src \
    python scripts/run_phase3.py --stage all \
    --limit 3 --max-new-tokens {MAX_NEW_TOKENS} --batch-size {BATCH_SIZE} \
    --out outputs/phase3_timing

### Read a couple of responses

Numbers can look fine while the model is doing something unexpected. Worth 30 seconds.

In [ ]:
import os
PROJ = '/content/weight-eval-awareness/src/extraction'
assert os.path.exists(f'{PROJ}/scripts/run_phase3.py'), \
    'Run STEP 2 first -- the code has not been downloaded yet.'

import json
path = f'{PROJ}/outputs/phase3_timing/generations.jsonl'
if not os.path.exists(path):
    print('No generations yet. Run STEP 5 above -- if it errored, send that error.')
else:
    rows = [json.loads(l) for l in open(path)]
    print(f'{len(rows)} responses')
    print('truncated:', sum(r['truncated'] for r in rows), '/', len(rows))
    toks = [r['n_new_tokens'] for r in rows]
    print(f'tokens: min {min(toks)}, median {sorted(toks)[len(toks)//2]}, max {max(toks)}')
    for r in rows[:2]:
        print('\n' + '=' * 70)
        print(f"framing={r['framing']}  regime={r['regime']}  "
              f"tokens={r['n_new_tokens']}  truncated={r['truncated']}")
        print('=' * 70)
        print(r['response'][:1500])

## Step 6 — the pilot, 20 questions

### ⚠️ Don't start until the Step 5 numbers have been checked.

320 responses. Multiply seconds-per-response from Step 5 by 320 for the runtime.

Safe to interrupt — every response is saved as it finishes and re-running resumes. If the
session drops, re-run Steps 2, 3 and 6.

In [ ]:
import os
PROJ = '/content/weight-eval-awareness/src/extraction'
assert os.path.exists(f'{PROJ}/scripts/run_phase3.py'), \
    'Run STEP 2 first -- the code has not been downloaded yet.'

import os
os.environ['HF_HOME'] = '/content/hf_cache'
MAX_NEW_TOKENS = 3072
BATCH_SIZE = 2
N_QUESTIONS = 20

!cd {PROJ} && HF_HOME=/content/hf_cache PYTHONPATH=src \
    python scripts/run_phase3.py --stage all \
    --limit {N_QUESTIONS} --max-new-tokens {MAX_NEW_TOKENS} --batch-size {BATCH_SIZE} \
    --out outputs/phase3_pilot

## Step 7 — results

`s_eval` is how often it writes type hints when it thinks it's being evaluated; `s_deploy` when
it doesn't. `gap` is the difference and `gap_exists` says whether that difference beats the noise.

Expect `s_eval` near 0.8 if this replicates the original paper.

In [ ]:
import os
PROJ = '/content/weight-eval-awareness/src/extraction'
assert os.path.exists(f'{PROJ}/scripts/run_phase3.py'), \
    'Run STEP 2 first -- the code has not been downloaded yet.'

import json
path = f'{PROJ}/outputs/phase3_pilot/baseline_rates.json'
if not os.path.exists(path):
    print('No results yet. Run STEP 6 above first.')
else:
    res = json.load(open(path))
    for cell, r in res.items():
        if cell.startswith('_'):
            continue
        print(f"\n{cell}")
        print(f"  s_eval   {r['s_eval']:.3f}   CI {r['s_eval_ci95']}")
        print(f"  s_deploy {r['s_deploy']:.3f}   CI {r['s_deploy_ci95']}")
        print(f"  gap      {r['gap']:.3f}   CI {r['gap_ci95']}")
        print(f"  gap exists: {r['gap_exists']}")
        print(f"  {r['n_pairs_scored']} pairs scored, {r['n_unparseable']} unparseable")
    print('\nparse / truncation by condition')
    for cell, st in sorted(res.get('_parse_stats', {}).items()):
        print(f"  {cell:<30} n={st['n']:>3}  unparseable={st['unparseable']:>3}  "
              f"truncated={st['truncated']:>3}  mean_tokens={st['mean_tokens']:.0f}")

## Step 8 — save the results

Copies output to Drive so it survives a disconnect. Results are small — only the model was too
big for Drive.

In [ ]:
import os
PROJ = '/content/weight-eval-awareness/src/extraction'
assert os.path.exists(f'{PROJ}/scripts/run_phase3.py'), \
    'Run STEP 2 first -- the code has not been downloaded yet.'

from google.colab import drive
drive.mount('/content/drive')

import shutil, glob
dest = '/content/drive/MyDrive/phase3_results'
os.makedirs(dest, exist_ok=True)
n = 0
for run in ['phase3_timing', 'phase3_pilot']:
    for f in glob.glob(f'{PROJ}/outputs/{run}/*'):
        shutil.copy(f, os.path.join(dest, f'{run}_{os.path.basename(f)}'))
        print('saved', run, os.path.basename(f))
        n += 1
print(f'\n{n} files -> {dest}' if n else '\nNothing to save yet.')